# Phase 1 — NHTSA Recall Data Ingestion and Cleaning

In [1]:
import pandas as pd
import zipfile
import numpy as np
import requests
from pathlib import Path

In [2]:
Path('../data/raw').mkdir(parents=True, exist_ok=True)

url = "https://static.nhtsa.gov/odi/ffdd/rcl/FLAT_RCL_POST_2010.zip"
save_path = Path('../data/raw/FLAT_RCL_POST_2010.zip')

response = requests.get(url)

with open(save_path, 'wb') as f:
    f.write(response.content)

print(f"Saved {len(response.content) / 1e6:.1f} MB to {save_path}")

Saved 14.6 MB to ../data/raw/FLAT_RCL_POST_2010.zip


In [3]:
with zipfile.ZipFile(save_path, 'r') as z:
    z.extractall('../data/raw')

In [4]:
text_path = Path('../data/raw/FLAT_RCL_POST_2010.txt')
df_raw = pd.read_csv(
    text_path,
    sep='\t',
    header=None,
    encoding='latin-1',
    low_memory=False,
    on_bad_lines='skip'
)
print(df_raw.shape)


(241044, 29)


## Load raw data

The NHTSA flat file is tab-delimited with no header row. The official 
data dictionary lists 26 columns, but the actual file contains 29 — 
the dictionary omits a leading record ID column and two trailing flag 
columns. Column names are assigned manually based on position.

In [5]:
columns = [
    'RECORD_ID', 'CAMPNO', 'MAKETXT', 'MODELTXT', 'YEARTXT', 'MFGCAMPNO',
    'COMPNAME', 'MFGNAME', 'BGMAN', 'ENDMAN', 'RCLTYPECD',
    'POTAFF', 'ODATE', 'INFLUENCED_BY', 'MFGTXT', 'RCDATE',
    'DATEA', 'RPNO', 'FMVSS', 'DESC_DEFECT', 'CONEQUENCE_DEFECT',
    'CORRECTIVE_ACTION', 'NOTES', 'RCL_CMPT_ID', 'MFR_COMP_NAME',
    'MFR_COMP_DESC', 'MFR_COMP_PTNO', 'POTENTIALLY_AFFECTED_FLAG',
    'COMPLETION_RATE'
]

In [6]:
df = df_raw.copy()
df.columns =columns
df. head(3)

,RECORD_ID,CAMPNO,MAKETXT,MODELTXT,YEARTXT,MFGCAMPNO,COMPNAME,MFGNAME,BGMAN,ENDMAN,...,DESC_DEFECT,CONEQUENCE_DEFECT,CORRECTIVE_ACTION,NOTES,RCL_CMPT_ID,MFR_COMP_NAME,MFR_COMP_DESC,MFR_COMP_PTNO,POTENTIALLY_AFFECTED_FLAG,COMPLETION_RATE
0,81715,10V484000,CARRIAGE,CAMEO,2004,NaN,EQUIPMENT,"CARRIAGE, INC",NaN,NaN,...,CARRIAGE IS RECALLING CERTAIN RECREATIONAL VEH...,AN OVERHEATED RECEIVER COULD CAUSE A FIRE.,CARRIAGE IS WORKING WITH DIMPLEX AND DIMPLEX W...,OWNERS MAY ALSO CONTACT THE NATIONAL HIGHWAY T...,000037779000201430000000329,NaN,NaN,NaN,No,No
1,81716,10V531000,LANCE,992,2011,NaN,EQUIPMENT:RECREATIONAL VEHICLE/TRAILER,LANCE CAMPER MFG. CORP.,20100701.0,NaN,...,LANCE IS RECALLING CERTAIN MODEL YEAR 2011 REC...,A PROPANE LEAK IN THE PRESENCE OF AN IGNITION ...,LANCE IS WORKING WITH ATWOOD AND WILL REPLACE ...,OWNERS MAY ALSO CONTACT THE NATIONAL HIGHWAY T...,000038037000992769000000330,NaN,NaN,NaN,No,No
2,81717,10V531000,LANCE,1685,2011,NaN,EQUIPMENT:RECREATIONAL VEHICLE/TRAILER,LANCE CAMPER MFG. CORP.,20100701.0,NaN,...,LANCE IS RECALLING CERTAIN MODEL YEAR 2011 REC...,A PROPANE LEAK IN THE PRESENCE OF AN IGNITION ...,LANCE IS WORKING WITH ATWOOD AND WILL REPLACE ...,OWNERS MAY ALSO CONTACT THE NATIONAL HIGHWAY T...,000038037000992770000000330,NaN,NaN,NaN,No,No


In [7]:
df_vehicle = df[df['RCLTYPECD'] == 'V'].copy()
print(df_vehicle.shape)
print(df_vehicle['RCDATE'].dtype)
print(df_vehicle['RCDATE'].min(), df_vehicle['RCDATE'].max())

(214458, 29)
int64
20100101 20260601


In [8]:
# Convert 'RCDATE' from integer to proper datetime

df_vehicle['RCDATE'] = pd.to_datetime(df_vehicle['RCDATE'].astype(str), format='%Y%m%d', errors='coerce')
print(df_vehicle['RCDATE'].dtype)
print(df_vehicle['RCDATE'].min(), df_vehicle['RCDATE'].max())
df_vehicle['RCDATE'].head(10)

datetime64[ns]
2010-01-01 00:00:00 2026-06-01 00:00:00


0   2010-10-14
1   2010-11-01
2   2010-11-01
3   2010-11-01
4   2010-11-01
5   2010-11-01
6   2010-10-27
7   2010-10-27
8   2010-10-27
9   2010-10-27
Name: RCDATE, dtype: datetime64[ns]

In [9]:
# Create new column 'RECALL_YEAR'.
# This will serve as the primary time axis in Power BI

df_vehicle['RECALL_YEAR'] = df_vehicle['RCDATE'].dt.year

## Manufacturer name normalisation

The MFGNAME field contains raw manufacturer names entered inconsistently 
— the same OEM appears under multiple spellings and legal entity names. 
This step maps all variants to a single canonical name per OEM, which is 
essential for accurate grouping in the dashboard.

In [10]:
print(df_vehicle['MFGNAME'].nunique())
print(df_vehicle['MFGNAME'].value_counts())

1267
MFGNAME
Mercedes-Benz USA, LLC                      44181
Toyota Motor Engineering & Manufacturing    13840
Ford Motor Company                          12271
Honda (American Honda Motor Co.)            11493
PACCAR Incorporated                          5715
                                            ...  
OEM Systems, LLC                                1
Morgan Motor Company Limited                    1
A&J Vans Inc                                    1
McClain Trailers, Inc.                          1
AMERICAN MANUFACTURING OPERATIONS, INC.         1
Name: count, Length: 1267, dtype: int64


In [11]:
mb_variants = df_vehicle[df_vehicle['MFGNAME'].str.contains('Mercedes', case=False, na=False)]['MFGNAME'].value_counts()
mb_variants

MFGNAME
Mercedes-Benz USA, LLC                   44181
Mercedes-Benz USA, LLC.                   1694
MERCEDES-BENZ USA, LLC.                     45
MERCEDES-BENZ USA, LLC - DBA SPRINTER       24
Mercedes-Benz USA, LLC - DBA Sprinter        8
Name: count, dtype: int64

In [16]:
df_vehicle['MFGNAME'].value_counts().head(60)

MFGNAME
Mercedes-Benz USA, LLC                      44181
Toyota Motor Engineering & Manufacturing    13840
Ford Motor Company                          12271
Honda (American Honda Motor Co.)            11493
PACCAR Incorporated                          5715
Forest River, Inc.                           4658
Daimler Trucks North America, LLC            4540
BMW of North America, LLC                    3801
Volkswagen Group of America, Inc.            3790
Daimler Trucks North America LLC             3735
Jaguar Land Rover North America, LLC         3570
Nissan North America, Inc.                   2953
Prevost Car (US) Inc.                        2792
Daimler Vans USA, LLC                        2519
General Motors LLC                           2331
New Flyer of America, Inc.                   2327
Porsche Cars North America, Inc.             2289
General Motors, LLC                          2014
Jayco, Inc.                                  1987
Hyundai Motor America                     